In [ ]:
%ls
import glob
import os
import cv2
import numpy as np


from driver import io_shape_dict
from driver_base import FINNExampleOverlay
from utils import letterbox, scale_coords
from driver import io_shape_dict
from driver_base import FINNExampleOverlay

coco_evaluate.py  driver.py      __pycache__/  scale.npy
coco_prep.py      finn/          qonnx/        testing_on_fpga.ipynb
data/             loaded.xclbin  resizer.bit   utils.py
driver_base.py    models.py      resizer.hwh


In [ ]:
driver = FINNExampleOverlay(
    bitfile_name="./finn-accel.bit",
    platform="zynq-iodma",
    io_shape_dict=io_shape_dict,
    batch_size=1,
    runtime_weight_dir="runtime_weights/",
)

In [ ]:
strides = [16, 32]
anchors_config = [
    [[10,14], [23,27], [37,58]],     # P4
    [[81,82], [135,169], [344,319]]  # P5
]
nc = 1
scale1 = np.load("./mul_0.npy")
scale2 = np.load("./mul_1.npy")
class_names = ['Person']

In [9]:
source = "/home/xilinx/jupyter_notebooks/Pynq-Z1/data/inputs/"
output_folder = "test_outputs/"

In [12]:
!ls /home/xilinx/jupyter_notebooks/Pynq-Z1/data/inputs/

010_jpg.rf.7f18943dcaabf6e9b39f8b2edb5e4268.jpg
0_193_jpg.rf.b77aaf10df5d22a0796b780a85a1be80.jpg
01_jpg.rf.193c13eb36f64e932b62f142230fdb65.jpg
1_027_jpg.rf.7977159e7c586c896cbd8da31187cda8.jpg
1_036_jpg.rf.70009e4be82b29238c6a580fec08744d.jpg
2010_09_018_067_jpg.rf.aa65ecb7900583827979dfa08ee5c180.jpg
2016031508000003_jpg.rf.8991a80de087e08020fcd7ccadcf81e1.jpg
2016031508000004_jpg.rf.deb1f08382334d7e8f7d010ca6b8e2c0.jpg
2016031508000005_jpg.rf.bcc9af055a05852faa889bb25b35cf08.jpg
2016031508000010_jpg.rf.1a794ee509a3ef4215a6715d9cef9c62.jpg
2016031508000013_jpg.rf.6a15b85e8c44776ecc6bc7622cda7a29.jpg
2016031508000034_jpg.rf.c9d16d53850cb0fb3b7eabe64060e566.jpg
2016031508000038_jpg.rf.2df17b3aff5bab441453c5b1208c9142.jpg
2016031508000042_jpg.rf.0130cd20ebf9c9ffaf511084686dcf15.jpg
2016031508000044_jpg.rf.5e16f721b4cb9573669190282c7dee31.jpg
2016031508000049_jpg.rf.f3dfd7f0049998033b9c81a1cde215c8.jpg
2016031508000057_jpg.rf.78c767df77fdcd6643e0a1e0277d1ec3.jpg
2016031509000003_jpg.rf.

crop001836_png_jpg.rf.21bb02ef9289cff180729d49db63c999.jpg
dsc_0475_jpg.rf.958de72bfc4778896b1b30b6ebe26ca4.jpg
dsc_1010_jpg.rf.6b75d3e034148740bd57a49864bdf443.jpg
gt_eval.json
img_0001_jpg.rf.4f40bf8f75a965aee2246072765e1e5e.jpg
img_0002_jpg.rf.fc6cedb86e4ac04125bbb2f64b664616.jpg
img_0003_jpg.rf.2bae6d53424fb485c6552765336f0ee7.jpg
img_0003_jpg.rf.5a22abaf8267354dd6de9ecd91119e32.jpg
img_0003_png_jpg.rf.09d4bf68aec8612cb940065cfb1e2dc3.jpg
img_0006_jpg.rf.7218dd0ce142e17ef5c7c3296e1a6dc2.jpg
img_0007_jpg.rf.7da27fbcf1b56c15ed11cd42454da4fc.jpg
img_0008_jpg.rf.e445cbc825cae2b20b87b57f7db4811f.jpg
img_0010_jpg.rf.8a68e64069f5f592e4d73f295b8862f1.jpg
img_0011_jpg.rf.41ae35b886fc54d7683eab15aef4a9ad.jpg
img_0013_jpg.rf.699b55fdd8c6032f52f5f15087366b29.jpg
img_0013_jpg.rf.d44e34565d6a8d122cb328281a84f912.jpg
img_0015_jpg.rf.fc02a567350439a96f46881c9c9d7e47.jpg
img_0016_jpg.rf.0ff83a6a69062ab80725b953ab4ba0b3.jpg
img_0016_jpg.rf.3359709db9870f2f5a020dda5f0386e4.jpg
img_0018_jpg.rf.69aef70

In [13]:
img_paths = glob.glob(source + "**/*.jpg", recursive=True)

In [14]:
len_img_paths = len(img_paths)
print(len_img_paths)

1882


In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [ ]:
def postprocess_yolo_outputs(outputs, anchors_config, strides, nc):
    """
    Post-process YOLO outputs from your FINN-exported model.

    Args:
        outputs: list of tensors [output_P4, output_P5]
                 shapes: [(1, 18, 26, 26), (1, 18, 13, 13)]
        anchors_config: list of anchor boxes per layer, [[na,2], [na,2]]
        strides: list of strides per layer, e.g. [16, 32]
        nc: number of classes

    Returns:
        predictions: [batch, num_predictions, nc+5] in format [x, y, w, h, obj, cls...]
    """
    z = []

    # Loop over layers
    for idx, output in enumerate(outputs):
        bs, _, ny, nx = output.shape  # e.g., (1, 18, 26, 26)
        anchors = np.array(anchors_config[idx], dtype=np.float32)  # [na, 2]
        stride = strides[idx]
        na = len(anchors)
        no = nc + 5

        # Reshape to [batch, na, no, ny, nx]
        output = output.reshape(bs, na, no, ny, nx)
        # Transpose to [batch, na, ny, nx, no]
        output = output.transpose(0, 1, 3, 4, 2)

        # Create grid
        yv, xv = np.meshgrid(np.arange(ny), np.arange(nx), indexing='ij')
        grid = np.stack((xv, yv), axis=2).reshape(1, 1, ny, nx, 2).astype(np.float32)

        # Create anchor grid
        anchor_grid = anchors.reshape(1, na, 1, 1, 2)

        # Sigmoid activation
        y_sigmoid = sigmoid(output)

        # xy coordinates (center) in absolute image scale
        xy = (y_sigmoid[..., 0:2] * 2.0 - 0.5 + grid) * stride
        # width/height in absolute scale
        wh = (y_sigmoid[..., 2:4] * 2.0) ** 2 * anchor_grid
        # objectness and class scores
        obj_cls = y_sigmoid[..., 4:]

        # Concatenate to [x_center, y_center, w, h, obj, cls...]
        y_out = np.concatenate([xy, wh, obj_cls], axis=-1)

        # Flatten to [batch, num_predictions, no]
        z.append(y_out.reshape(bs, -1, no))

    # Combine both layers
    return np.concatenate(z, axis=1)

In [ ]:
def box_iou(box1, box2):
    """
    Compute IoU between two sets of boxes.
    Boxes are in [x1, y1, x2, y2] format.
    """
    # Expand dimensions for broadcasting
    box1 = np.expand_dims(box1, axis=1)  # [N, 1, 4]
    box2 = np.expand_dims(box2, axis=0)  # [1, M, 4]
    
    # Intersection coordinates
    inter_x1 = np.maximum(box1[..., 0], box2[..., 0])
    inter_y1 = np.maximum(box1[..., 1], box2[..., 1])
    inter_x2 = np.minimum(box1[..., 2], box2[..., 2])
    inter_y2 = np.minimum(box1[..., 3], box2[..., 3])
    
    # Intersection area
    inter_w = np.maximum(0, inter_x2 - inter_x1)
    inter_h = np.maximum(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h
    
    # Union area
    box1_area = (box1[..., 2] - box1[..., 0]) * (box1[..., 3] - box1[..., 1])
    box2_area = (box2[..., 2] - box2[..., 0]) * (box2[..., 3] - box2[..., 1])
    union_area = box1_area + box2_area - inter_area
    
    # IoU
    iou = inter_area / (union_area + 1e-6)
    return iou

In [ ]:
def xywh2xyxy(boxes):
    """Convert boxes from [x_center, y_center, w, h] to [x1, y1, x2, y2]."""
    xy = boxes[..., :2]
    wh = boxes[..., 2:4]
    xy1 = xy - wh / 2
    xy2 = xy + wh / 2
    return np.concatenate([xy1, xy2], axis=-1)

In [ ]:
def non_max_suppression(prediction, conf_thres=0.25, iou_thres=0.45):
    """
    Performs Non-Maximum Suppression (NMS) on inference results.
    Based on utils.general.non_max_suppression from your code.
    
    Returns:
         list of detections, on (n,6) tensor per image [xyxy, conf, cls]
    """
    xc = prediction[..., 4] > conf_thres  # candidates
    
    # Settings
    max_wh = 7680  # maximum box width and height
    max_nms = 30000  # maximum number of boxes into torchvision.ops.nms()
    
    output = []
    for xi, x in enumerate(prediction):  # image index, image inference
        # Apply constraints
        x = x[xc[xi]]  # confidence
        
        # If none remain process next image
        if not x.shape[0]:
            output.append(np.zeros((0, 6)))
            continue
        
        # Compute conf
        x[:, 5:] *= x[:, 4:5]  # conf = obj_conf * cls_conf
        
        # Box (center x, center y, width, height) to (x1, y1, x2, y2)
        box = xywh2xyxy(x[:, :4])
        
        # Detections matrix nx6 (xyxy, conf, cls)
        conf = np.max(x[:, 5:], axis=1, keepdims=True)
        j = np.argmax(x[:, 5:], axis=1).reshape(-1, 1)
        x = np.concatenate((box, conf, j.astype(np.float32)), axis=1)[conf.reshape(-1) > conf_thres]
        
        # Check shape
        n = x.shape[0]  # number of boxes
        if not n:
            output.append(np.zeros((0, 6)))
            continue
        
        # Sort by confidence
        x = x[x[:, 4].argsort()[::-1][:max_nms]]
        
        # Batched NMS
        c = x[:, 5:6] * max_wh  # classes
        boxes, scores = x[:, :4] + c, x[:, 4]  # boxes (offset by class), scores
        
        # NMS
        keep = []
        indices = np.arange(x.shape[0])
        
        while boxes.shape[0]:
            # Pick detection with highest confidence (already sorted, so index 0)
            keep.append(indices[0])
            
            if boxes.shape[0] == 1:
                break
            
            # Compute IoU with remaining boxes
            ious = box_iou(boxes[0:1], boxes[1:])
            
            # Remove boxes with high IoU
            mask = ious[0] < iou_thres
            boxes = boxes[1:][mask]
            scores = scores[1:][mask]
            indices = indices[1:][mask]
        
        if keep:
            keep_indices = np.array(keep)
            output.append(x[keep_indices])
        else:
            output.append(np.zeros((0, 6)))
    
    return output

In [ ]:
def draw_detections(image, detections, class_names=None):
    """Draw bounding boxes on image."""
    image_draw = image.copy()
    
    for det in detections:
        x1, y1, x2, y2, conf, cls = det
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        cls = int(cls)
        
        # Draw box
        color = (0, 255, 0)
        cv2.rectangle(image_draw, (x1, y1), (x2, y2), color, 2)
        
        # Draw label
        label = f"{class_names[cls] if class_names else cls}: {conf:.2f}"
        (label_w, label_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
        cv2.rectangle(image_draw, (x1, y1 - label_h - 10), (x1 + label_w, y1), color, -1)
        cv2.putText(image_draw, label, (x1, y1 - 5),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)
    
    return image_draw

In [ ]:
img_size = (416, 416)
conf_thres = 0.05
iou_thres = 0.5

for number, test_img_path in enumerate(img_paths):
    print(f"[INFO] processing {number+1}/{len_img_paths}")
    img_org = cv2.imread(test_img_path)
    img = img_org.copy()

    h, w, _ = img_org.shape
    img, ratio, (dw, dh) = letterbox(img, img_size, auto=False)

    img = img[:, :, ::-1]
    img = img.astype(np.uint8)

    driver_in = np.expand_dims(img, 0)

    output = driver.execute(driver_in)
    output1 = output[0].transpose(0, 3, 1, 2) 
    output2 = output[1].transpose(0, 3, 1, 2) 
    
    # Scaling
    output1 = output1 * scale1
    output2 = output2 * scale2
    
    outputs = [output2, output1]
    
    # Post-processing
    pred = postprocess_yolo_outputs(
        outputs,
        anchors_config=anchors_config,
        strides=strides,
        nc=nc
    )
    
    print(f"Prediction shape: {pred.shape}")
    print(f"Number of anchor prediction: {pred.shape[1]}")
    
    # Apply NMS
    print("Applying NMS...")
    print(f"Number of predictions > {conf_thres}: {(pred[..., 4] > conf_thres).sum()}")
    
    pred = non_max_suppression(pred, conf_thres, iou_thres)
    
    # Get detected boxes_detected, labels, confidences, class-scores.
    
    boxes_detected, class_names_detected, probs_detected = [], [], []

    folder_name, file_name = test_img_path.split("/")[-2:]
    folder_name = folder_name.replace('images', 'labels') 
    save_folder = output_folder + folder_name
    
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
    save_path = save_folder + "/" + file_name.replace(".jpg", ".txt")
    f = open(save_path, "w")
    det = pred[0] if len(pred)>0 else []

    # Process predictions
    if len(det):
        f.write(file_name+"\n")
        f.write(str(len(det))+"\n")
        # Rescale boxes from img_size to im0 size
        det[:, :4] = scale_coords(img.shape, det[:, :4], img_org.shape).round() # type: ignore
        
        # Draw and save result
        result_image = draw_detections(img, det, class_names)
        output_path = f'output_detection_{number+1}.jpg'
        cv2.imwrite(output_path, result_image)
        print(f"\nResult saved to {output_path}")
        
        for i, d in enumerate(det):
            x1, y1, x2, y2, conf, cls = det
            class_name = class_names[int(cls)] if class_names else int(cls)
            print(f"  Detection {i+1}: {class_name} ({conf:.3f}) at [{x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f}]")
            f.write("%.5f %.5f %.5f %.5f %g\n"%(d[0], d[1], d[2]-d[0], d[3]-d[1], d[4]))
    f.close()
    

[INFO] processing 1/1882
[INFO] processing 2/1882
[INFO] processing 3/1882
[INFO] processing 4/1882
[INFO] processing 5/1882
[INFO] processing 6/1882
[INFO] processing 7/1882
[INFO] processing 8/1882
[INFO] processing 9/1882
[INFO] processing 10/1882
[INFO] processing 11/1882
[INFO] processing 12/1882
[INFO] processing 13/1882
[INFO] processing 14/1882
[INFO] processing 15/1882
[INFO] processing 16/1882
[INFO] processing 17/1882
[INFO] processing 18/1882
[INFO] processing 19/1882
[INFO] processing 20/1882
[INFO] processing 21/1882
[INFO] processing 22/1882
[INFO] processing 23/1882
[INFO] processing 24/1882
[INFO] processing 25/1882
[INFO] processing 26/1882
[INFO] processing 27/1882
[INFO] processing 28/1882
[INFO] processing 29/1882
[INFO] processing 30/1882
[INFO] processing 31/1882
[INFO] processing 32/1882
[INFO] processing 33/1882
[INFO] processing 34/1882
[INFO] processing 35/1882
[INFO] processing 36/1882
[INFO] processing 37/1882
[INFO] processing 38/1882
[INFO] processing 39/

[INFO] processing 310/1882
[INFO] processing 311/1882
[INFO] processing 312/1882
[INFO] processing 313/1882
[INFO] processing 314/1882
[INFO] processing 315/1882
[INFO] processing 316/1882
[INFO] processing 317/1882
[INFO] processing 318/1882
[INFO] processing 319/1882
[INFO] processing 320/1882
[INFO] processing 321/1882
[INFO] processing 322/1882
[INFO] processing 323/1882
[INFO] processing 324/1882
[INFO] processing 325/1882
[INFO] processing 326/1882
[INFO] processing 327/1882
[INFO] processing 328/1882
[INFO] processing 329/1882
[INFO] processing 330/1882
[INFO] processing 331/1882
[INFO] processing 332/1882
[INFO] processing 333/1882
[INFO] processing 334/1882
[INFO] processing 335/1882
[INFO] processing 336/1882
[INFO] processing 337/1882
[INFO] processing 338/1882
[INFO] processing 339/1882
[INFO] processing 340/1882
[INFO] processing 341/1882
[INFO] processing 342/1882
[INFO] processing 343/1882
[INFO] processing 344/1882
[INFO] processing 345/1882
[INFO] processing 346/1882
[

[INFO] processing 614/1882
[INFO] processing 615/1882
[INFO] processing 616/1882
[INFO] processing 617/1882
[INFO] processing 618/1882
[INFO] processing 619/1882
[INFO] processing 620/1882
[INFO] processing 621/1882
[INFO] processing 622/1882
[INFO] processing 623/1882
[INFO] processing 624/1882
[INFO] processing 625/1882
[INFO] processing 626/1882
[INFO] processing 627/1882
[INFO] processing 628/1882
[INFO] processing 629/1882
[INFO] processing 630/1882
[INFO] processing 631/1882
[INFO] processing 632/1882
[INFO] processing 633/1882
[INFO] processing 634/1882
[INFO] processing 635/1882
[INFO] processing 636/1882
[INFO] processing 637/1882
[INFO] processing 638/1882
[INFO] processing 639/1882
[INFO] processing 640/1882
[INFO] processing 641/1882
[INFO] processing 642/1882
[INFO] processing 643/1882
[INFO] processing 644/1882
[INFO] processing 645/1882
[INFO] processing 646/1882
[INFO] processing 647/1882
[INFO] processing 648/1882
[INFO] processing 649/1882
[INFO] processing 650/1882
[

[INFO] processing 918/1882
[INFO] processing 919/1882
[INFO] processing 920/1882
[INFO] processing 921/1882
[INFO] processing 922/1882
[INFO] processing 923/1882
[INFO] processing 924/1882
[INFO] processing 925/1882
[INFO] processing 926/1882
[INFO] processing 927/1882
[INFO] processing 928/1882
[INFO] processing 929/1882
[INFO] processing 930/1882
[INFO] processing 931/1882
[INFO] processing 932/1882
[INFO] processing 933/1882
[INFO] processing 934/1882
[INFO] processing 935/1882
[INFO] processing 936/1882
[INFO] processing 937/1882
[INFO] processing 938/1882
[INFO] processing 939/1882
[INFO] processing 940/1882
[INFO] processing 941/1882
[INFO] processing 942/1882
[INFO] processing 943/1882
[INFO] processing 944/1882
[INFO] processing 945/1882
[INFO] processing 946/1882
[INFO] processing 947/1882
[INFO] processing 948/1882
[INFO] processing 949/1882
[INFO] processing 950/1882
[INFO] processing 951/1882
[INFO] processing 952/1882
[INFO] processing 953/1882
[INFO] processing 954/1882
[

[INFO] processing 1214/1882
[INFO] processing 1215/1882
[INFO] processing 1216/1882
[INFO] processing 1217/1882
[INFO] processing 1218/1882
[INFO] processing 1219/1882
[INFO] processing 1220/1882
[INFO] processing 1221/1882
[INFO] processing 1222/1882
[INFO] processing 1223/1882
[INFO] processing 1224/1882
[INFO] processing 1225/1882
[INFO] processing 1226/1882
[INFO] processing 1227/1882
[INFO] processing 1228/1882
[INFO] processing 1229/1882
[INFO] processing 1230/1882
[INFO] processing 1231/1882
[INFO] processing 1232/1882
[INFO] processing 1233/1882
[INFO] processing 1234/1882
[INFO] processing 1235/1882
[INFO] processing 1236/1882
[INFO] processing 1237/1882
[INFO] processing 1238/1882
[INFO] processing 1239/1882
[INFO] processing 1240/1882
[INFO] processing 1241/1882
[INFO] processing 1242/1882
[INFO] processing 1243/1882
[INFO] processing 1244/1882
[INFO] processing 1245/1882
[INFO] processing 1246/1882
[INFO] processing 1247/1882
[INFO] processing 1248/1882
[INFO] processing 12

[INFO] processing 1508/1882
[INFO] processing 1509/1882
[INFO] processing 1510/1882
[INFO] processing 1511/1882
[INFO] processing 1512/1882
[INFO] processing 1513/1882
[INFO] processing 1514/1882
[INFO] processing 1515/1882
[INFO] processing 1516/1882
[INFO] processing 1517/1882
[INFO] processing 1518/1882
[INFO] processing 1519/1882
[INFO] processing 1520/1882
[INFO] processing 1521/1882
[INFO] processing 1522/1882
[INFO] processing 1523/1882
[INFO] processing 1524/1882
[INFO] processing 1525/1882
[INFO] processing 1526/1882
[INFO] processing 1527/1882
[INFO] processing 1528/1882
[INFO] processing 1529/1882
[INFO] processing 1530/1882
[INFO] processing 1531/1882
[INFO] processing 1532/1882
[INFO] processing 1533/1882
[INFO] processing 1534/1882
[INFO] processing 1535/1882
[INFO] processing 1536/1882
[INFO] processing 1537/1882
[INFO] processing 1538/1882
[INFO] processing 1539/1882
[INFO] processing 1540/1882
[INFO] processing 1541/1882
[INFO] processing 1542/1882
[INFO] processing 15

[INFO] processing 1802/1882
[INFO] processing 1803/1882
[INFO] processing 1804/1882
[INFO] processing 1805/1882
[INFO] processing 1806/1882
[INFO] processing 1807/1882
[INFO] processing 1808/1882
[INFO] processing 1809/1882
[INFO] processing 1810/1882
[INFO] processing 1811/1882
[INFO] processing 1812/1882
[INFO] processing 1813/1882
[INFO] processing 1814/1882
[INFO] processing 1815/1882
[INFO] processing 1816/1882
[INFO] processing 1817/1882
[INFO] processing 1818/1882
[INFO] processing 1819/1882
[INFO] processing 1820/1882
[INFO] processing 1821/1882
[INFO] processing 1822/1882
[INFO] processing 1823/1882
[INFO] processing 1824/1882
[INFO] processing 1825/1882
[INFO] processing 1826/1882
[INFO] processing 1827/1882
[INFO] processing 1828/1882
[INFO] processing 1829/1882
[INFO] processing 1830/1882
[INFO] processing 1831/1882
[INFO] processing 1832/1882
[INFO] processing 1833/1882
[INFO] processing 1834/1882
[INFO] processing 1835/1882
[INFO] processing 1836/1882
[INFO] processing 18